# Lab 5: Heuristic functions with the 8-puzzle

## Learning goals
By the end of this lab, students can:
- formulate the 8-puzzle as a search problem;
- implement h1 as misplaced tiles and h2 as Manhattan distance;
- explain relaxed problems and why they can produce admissible heuristics;
- compare A* search with h(n)=0, h1, and h2;
- step through the search process and the final puzzle solution in a GUI.

Source note: The notes in this notebook are paraphrased from Chapter 3, Solving Problems by Searching, in the uploaded course document. No outside notes are used.


## Chapter 3 notes for this lab

The 8-puzzle is a standardized search problem. A state records the location of each numbered tile and the blank. Actions move the blank up, down, left, or right when possible. Each action costs 1. The goal is usually the ordered board.

Two common heuristics are:
- h1: number of misplaced tiles, not counting the blank;
- h2: sum of horizontal and vertical distances from each tile to its goal location, also called Manhattan distance.

These heuristics come from relaxed versions of the puzzle. If tiles could move directly to any square, h1 would be exact for that easier problem. If tiles could move one square at a time without needing the blank square, h2 would be exact for that easier problem. Because relaxed problems are easier than the original, their optimal costs are lower bounds and can be used as admissible heuristics.


In [1]:
# Run this cell first.
# These notebooks use only local Python code plus matplotlib and ipywidgets.
# If a student machine is missing a package, uncomment and run the next line once:
# %pip install matplotlib ipywidgets

import math
import heapq
import itertools
from collections import deque
from html import escape

import matplotlib.pyplot as plt

# HTML display works even if ipywidgets is not installed.
from IPython.display import display, HTML, clear_output

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception as exc:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Install with: pip install ipywidgets")


In [2]:
_NODE_COUNTER = itertools.count()

def reset_node_counter():
    global _NODE_COUNTER
    _NODE_COUNTER = itertools.count()

def make_html_table(headers, rows, max_rows=15):
    rows = rows[:max_rows]
    html = "<table style='border-collapse:collapse; font-family:monospace;'>"
    html += "<tr>" + "".join(f"<th style='border:1px solid #999; padding:4px 8px;'>{escape(str(h))}</th>" for h in headers) + "</tr>"
    for row in rows:
        html += "<tr>" + "".join(f"<td style='border:1px solid #999; padding:4px 8px;'>{escape(str(v))}</td>" for v in row) + "</tr>"
    html += "</table>"
    return HTML(html)

def make_stepper(steps, draw_func, table_func=None, title="Step-through viewer"):
    if not steps:
        print("No steps to show.")
        return

    if not WIDGETS_AVAILABLE:
        print("ipywidgets is not available, so only the final step is shown.")
        draw_func(steps[-1])
        if table_func:
            display(table_func(steps[-1]))
        return

    slider = widgets.IntSlider(value=0, min=0, max=len(steps) - 1, step=1, description="Step")
    previous_button = widgets.Button(description="Previous")
    next_button = widgets.Button(description="Next")
    play = widgets.Play(value=0, min=0, max=len(steps) - 1, step=1, interval=1200, description="Play")
    widgets.jslink((play, "value"), (slider, "value"))
    out = widgets.Output()

    def render(*args):
        with out:
            clear_output(wait=True)
            step = steps[slider.value]
            draw_func(step)
            if table_func:
                display(table_func(step))

    def go_previous(_):
        slider.value = max(slider.min, slider.value - 1)

    def go_next(_):
        slider.value = min(slider.max, slider.value + 1)

    previous_button.on_click(go_previous)
    next_button.on_click(go_next)
    slider.observe(render, names="value")
    display(HTML(f"<h3>{escape(title)}</h3>"))
    display(widgets.HBox([previous_button, next_button, play, slider]))
    display(out)
    render()


In [3]:
# 8-puzzle representation:
# A state is a tuple of 9 numbers. The blank is 0.
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

def board_rows(state):
    return [state[0:3], state[3:6], state[6:9]]

def puzzle_actions(state):
    blank = state.index(0)
    r, c = divmod(blank, 3)
    moves = []
    for action, (dr, dc) in [
        ("Up", (-1, 0)),
        ("Down", (1, 0)),
        ("Left", (0, -1)),
        ("Right", (0, 1)),
    ]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_blank = nr * 3 + nc
            new_state = list(state)
            new_state[blank], new_state[new_blank] = new_state[new_blank], new_state[blank]
            moves.append((action, tuple(new_state)))
    return moves

def misplaced_tiles(state):
    return sum(1 for i, tile in enumerate(state) if tile != 0 and tile != GOAL[i])

GOAL_POS = {tile: divmod(i, 3) for i, tile in enumerate(GOAL)}

def manhattan_distance(state):
    total = 0
    for i, tile in enumerate(state):
        if tile == 0:
            continue
        r, c = divmod(i, 3)
        gr, gc = GOAL_POS[tile]
        total += abs(r - gr) + abs(c - gc)
    return total

def h_zero(state):
    return 0

def is_solvable(state):
    numbers = [x for x in state if x != 0]
    inversions = 0
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] > numbers[j]:
                inversions += 1
    return inversions % 2 == 0

def scramble(goal=GOAL, moves=14, seed=7):
    import random
    random.seed(seed)
    state = goal
    previous = None
    opposite = {"Up": "Down", "Down": "Up", "Left": "Right", "Right": "Left"}
    for _ in range(moves):
        choices = [(a, s) for a, s in puzzle_actions(state) if previous is None or a != opposite.get(previous)]
        action, state = random.choice(choices)
        previous = action
    return state

START = scramble(moves=14, seed=10)
print("Start state:", START)
print("Solvable:", is_solvable(START))
print("h1 misplaced:", misplaced_tiles(START))
print("h2 Manhattan:", manhattan_distance(START))


Start state: (8, 1, 2, 4, 7, 3, 0, 6, 5)
Solvable: True
h1 misplaced: 7
h2 Manhattan: 12


In [4]:
reset_node_counter()

class PuzzleNode:
    def __init__(self, state, parent=None, action=None, path_cost=0, depth=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = depth
        self.order = next(_NODE_COUNTER)

def puzzle_path_nodes(node):
    nodes = []
    while node is not None:
        nodes.append(node)
        node = node.parent
    return list(reversed(nodes))

def puzzle_path_states(node):
    return [n.state for n in puzzle_path_nodes(node)]

def puzzle_path_actions(node):
    return [n.action for n in puzzle_path_nodes(node)[1:]]

def astar_puzzle_steps(start, heuristic, max_expansions=50000, label="A*"):
    reset_node_counter()
    root = PuzzleNode(start)
    frontier = [(root.path_cost + heuristic(root.state), root.order, root)]
    reached_cost = {start: 0}
    expanded = []
    steps = []
    generated_count = 1

    def live_frontier():
        live = []
        for f_value, _, node in frontier:
            if reached_cost.get(node.state) == node.path_cost:
                live.append(node)
        live.sort(key=lambda n: (n.path_cost + heuristic(n.state), n.order))
        return live

    def snapshot(message, current=None, generated=None, solution_node=None):
        steps.append({
            "step": len(steps),
            "algorithm": label,
            "message": message,
            "current": current,
            "generated": generated or [],
            "expanded_count": len(expanded),
            "generated_count": generated_count,
            "frontier_nodes": live_frontier()[:12],
            "solution_path": puzzle_path_states(solution_node) if solution_node else None,
        })

    snapshot("Initialize frontier with the start board.", current=root)

    while frontier and len(expanded) < max_expansions:
        _, _, node = heapq.heappop(frontier)
        if reached_cost.get(node.state) != node.path_cost:
            continue

        snapshot("Pop the board with the smallest f(n).", current=node)

        if node.state == GOAL:
            snapshot("Goal board reached.", current=node, solution_node=node)
            metrics = {
                "solution_depth": node.depth,
                "solution_cost": node.path_cost,
                "expanded_count": len(expanded),
                "generated_count": generated_count,
                "frontier_size": len(live_frontier()),
            }
            return node, steps, metrics

        expanded.append(node.state)
        generated_states = []
        for action, new_state in puzzle_actions(node.state):
            child = PuzzleNode(new_state, parent=node, action=action, path_cost=node.path_cost + 1, depth=node.depth + 1)
            if new_state not in reached_cost or child.path_cost < reached_cost[new_state]:
                reached_cost[new_state] = child.path_cost
                heapq.heappush(frontier, (child.path_cost + heuristic(new_state), child.order, child))
                generated_states.append(new_state)
                generated_count += 1
        snapshot("Expand current board and add improved children to the frontier.", current=node, generated=generated_states)

    snapshot("Search stopped before finding a solution. Increase max_expansions if needed.")
    return None, steps, {
        "solution_depth": None,
        "solution_cost": None,
        "expanded_count": len(expanded),
        "generated_count": generated_count,
        "frontier_size": len(live_frontier()),
    }


In [5]:
def draw_puzzle_board(state, ax, title=None):
    ax.set_xlim(0, 3)
    ax.set_ylim(0, 3)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    for i, tile in enumerate(state):
        r, c = divmod(i, 3)
        y = 2 - r
        face = "#f7f7f7" if tile != 0 else "#dddddd"
        rect = plt.Rectangle((c, y), 1, 1, facecolor=face, edgecolor="#333333", linewidth=2)
        ax.add_patch(rect)
        if tile != 0:
            ax.text(c + 0.5, y + 0.5, str(tile), ha="center", va="center", fontsize=22)
    if title:
        ax.set_title(title)

def draw_puzzle_step(step, heuristic=manhattan_distance, title="8-puzzle A* search"):
    current = step.get("current")
    fig, ax = plt.subplots(figsize=(4, 4))
    if current is not None:
        draw_puzzle_board(current.state, ax, title=title)
    else:
        draw_puzzle_board(GOAL, ax, title=title)
    plt.show()

def puzzle_step_html(step, heuristic=manhattan_distance):
    current = step.get("current")
    msg = escape(step.get("message", ""))
    text = f"<b>Step {step.get('step', 0)}.</b> {msg}<br>"
    if current is not None:
        f_value = current.path_cost + heuristic(current.state)
        text += f"<b>g(n):</b> {current.path_cost} | <b>h(n):</b> {heuristic(current.state)} | <b>f(n):</b> {f_value}<br>"
        actions = puzzle_path_actions(current)
        text += "<b>Current action path:</b> " + (" -> ".join(actions) if actions else "Start") + "<br>"
    text += f"<b>Expanded count:</b> {step.get('expanded_count', 0)} | <b>Generated count:</b> {step.get('generated_count', 0)}<br>"
    rows = []
    for node in step.get("frontier_nodes", []):
        rows.append([
            node.action,
            node.depth,
            node.path_cost,
            heuristic(node.state),
            node.path_cost + heuristic(node.state),
            node.state,
        ])
    table = ""
    if rows:
        table = make_html_table(["last action", "depth", "g(n)", "h(n)", "f(n)", "state"], rows, max_rows=12).data
    return HTML(text + table)

def draw_solution_move(step):
    state = step["state"]
    fig, ax = plt.subplots(figsize=(4, 4))
    draw_puzzle_board(state, ax, title=step["title"])
    plt.show()

def solution_step_html(step):
    text = f"<b>Move {step['move_index']}.</b> {escape(step['action'])}<br>"
    text += f"<b>h1 misplaced:</b> {misplaced_tiles(step['state'])} | <b>h2 Manhattan:</b> {manhattan_distance(step['state'])}"
    return HTML(text)

def solution_steps_from_node(node):
    nodes = puzzle_path_nodes(node)
    steps = []
    for i, n in enumerate(nodes):
        steps.append({
            "step": i,
            "move_index": i,
            "state": n.state,
            "action": "Start" if i == 0 else n.action,
            "title": "Start board" if i == 0 else f"After move {i}: {n.action}",
        })
    return steps

def effective_branching_factor(generated_nodes, depth, tolerance=1e-5):
    if depth <= 0:
        return 0.0
    # Solve generated_nodes + 1 = 1 + b + b^2 + ... + b^depth by binary search.
    target = generated_nodes + 1
    low, high = 1.0, max(2.0, generated_nodes)
    def series(b):
        return sum(b ** i for i in range(depth + 1))
    while series(high) < target:
        high *= 2
    for _ in range(80):
        mid = (low + high) / 2
        if series(mid) < target:
            low = mid
        else:
            high = mid
    return (low + high) / 2


## Compare A* using no heuristic, h1, and h2


In [6]:
heuristics = [
    ("h0: no heuristic", h_zero),
    ("h1: misplaced tiles", misplaced_tiles),
    ("h2: Manhattan distance", manhattan_distance),
]

puzzle_results = {}
rows = []
for name, h in heuristics:
    node, steps, metrics = astar_puzzle_steps(START, h, label=f"A* with {name}")
    puzzle_results[name] = (node, steps, metrics, h)
    depth = metrics["solution_depth"] or 0
    ebf = effective_branching_factor(metrics["generated_count"], depth) if depth else 0
    rows.append([
        name,
        metrics["solution_depth"],
        metrics["solution_cost"],
        metrics["expanded_count"],
        metrics["generated_count"],
        round(ebf, 3) if ebf else "",
    ])

make_html_table(["heuristic", "solution depth", "solution cost", "expanded", "generated", "effective b*"], rows)


heuristic,solution depth,solution cost,expanded,generated,effective b*
h0: no heuristic,14,14,3133,5117,1.731
h1: misplaced tiles,14,14,207,350,1.388
h2: Manhattan distance,14,14,22,42,1.138


## GUI 1: step through the A* search process


In [8]:
def puzzle_search_gui():
    if not WIDGETS_AVAILABLE:
        name = "h2: Manhattan distance"
        node, steps, metrics, h = puzzle_results[name]
        make_stepper(
            steps,
            draw_func=lambda step: draw_puzzle_step(step, heuristic=h, title=f"A* with {name}"),
            table_func=lambda step: puzzle_step_html(step, heuristic=h),
            title=f"A* with {name}"
        )
        return

    dropdown = widgets.Dropdown(options=[name for name, _ in heuristics], value="h2: Manhattan distance", description="Heuristic")
    run_button = widgets.Button(description="Show")
    out = widgets.Output()

    def on_run(_=None):
        with out:
            clear_output(wait=True)
            name = dropdown.value
            node, steps, metrics, h = puzzle_results[name]
            print("Start:", START)
            print("Goal :", GOAL)
            print("Solution depth:", metrics["solution_depth"])
            print("Expanded:", metrics["expanded_count"], "Generated:", metrics["generated_count"])
            make_stepper(
                steps,
                draw_func=lambda step: draw_puzzle_step(step, heuristic=h, title=f"A* with {name}"),
                table_func=lambda step: puzzle_step_html(step, heuristic=h),
                title=f"A* search process: {name}"
            )

    run_button.on_click(on_run)
    display(widgets.VBox([widgets.HBox([dropdown, run_button]), out]))
    on_run()

puzzle_search_gui()


## GUI 2: step through the final solution path


In [10]:
best_name = "h2: Manhattan distance"
solution_node, _, _, _ = puzzle_results[best_name]
solution_steps = solution_steps_from_node(solution_node)

make_stepper(
    solution_steps,
    draw_func=draw_solution_move,
    table_func=solution_step_html,
    title="8-puzzle final solution animation"
)

print("Actions:", " -> ".join(puzzle_path_actions(solution_node)))


Output()

Actions: Up -> Up -> Right -> Right -> Down -> Down -> Left -> Up -> Left -> Down -> Right -> Up -> Right -> Down


## Demonstrate dominance: h2 >= h1


In [11]:
# Test h2 >= h1 on many guaranteed-solvable boards created by scrambling the goal.
samples = [scramble(moves=m, seed=s) for m in range(1, 18) for s in range(5)]
dominance_checks = [(state, misplaced_tiles(state), manhattan_distance(state)) for state in samples]
violations = [row for row in dominance_checks if row[2] < row[1]]

if violations:
    print("Found a violation, which should not happen for these definitions:")
    print(violations[:3])
else:
    print("No violations found in samples: h2 dominates h1 for these boards.")

make_html_table(
    ["state", "h1 misplaced", "h2 Manhattan"],
    [[state, h1, h2] for state, h1, h2 in dominance_checks[:10]],
    max_rows=10
)


No violations found in samples: h2 dominates h1 for these boards.


state,h1 misplaced,h2 Manhattan
"(1, 2, 3, 4, 5, 6, 7, 0, 8)",1,1
"(1, 2, 3, 4, 5, 0, 7, 8, 6)",1,1
"(1, 2, 3, 4, 5, 0, 7, 8, 6)",1,1
"(1, 2, 3, 4, 5, 0, 7, 8, 6)",1,1
"(1, 2, 3, 4, 5, 0, 7, 8, 6)",1,1
"(1, 2, 3, 4, 5, 6, 0, 7, 8)",2,2
"(1, 2, 0, 4, 5, 3, 7, 8, 6)",2,2
"(1, 2, 0, 4, 5, 3, 7, 8, 6)",2,2
"(1, 2, 0, 4, 5, 3, 7, 8, 6)",2,2
"(1, 2, 3, 4, 0, 5, 7, 8, 6)",2,2


## Student practice


In [12]:
# Practice:
# 1. Change moves and seed to create a new puzzle.
# 2. Re-run the comparison table.
# 3. Which heuristic generates fewer nodes?

NEW_START = scramble(moves=10, seed=3)
print("New start:", NEW_START)
print("h1:", misplaced_tiles(NEW_START), "h2:", manhattan_distance(NEW_START))

for name, h in heuristics:
    node, steps, metrics = astar_puzzle_steps(NEW_START, h, label=f"A* with {name}")
    print(f"{name:25s} depth={metrics['solution_depth']} expanded={metrics['expanded_count']} generated={metrics['generated_count']}")


New start: (4, 1, 0, 7, 5, 2, 8, 6, 3)
h1: 7 h2: 10
h0: no heuristic          depth=10 expanded=463 generated=759
h1: misplaced tiles       depth=10 expanded=15 generated=28
h2: Manhattan distance    depth=10 expanded=10 generated=17
